In [1]:
import json
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

# Base URL of the website
base_url = 'https://datafranca.org'

# URL of the French AI terminologies page (first page)
start_url = 'https://datafranca.org/wiki/Cat%C3%A9gorie:GRAND_LEXIQUE_FRAN%C3%87AIS'

# Function to get the page content
def get_soup(url):
    response = requests.get(url)
    if response.status_code == 200:
        return BeautifulSoup(response.text, 'html.parser')
    else:
        print(f"Error: Unable to retrieve page content. Status code: {response.status_code}")
        return None

def get_translations(term_url):
    soup = get_soup(term_url)
    if not soup:
        return [], []  # Return empty lists if the page couldn't be fetched
    
    # Initialize lists for both French and English translations
    french_translations = []
    english_translations = []

    # Locate the "Français" section
    french_section = soup.find('h2', string='Français')
    if french_section:
        next_element = french_section.find_next_sibling()
        while next_element and next_element.name != 'h2':
            french_translations.append(next_element.text.strip())
            next_element = next_element.find_next_sibling()

    # Locate the "Anglais" section
    english_section = soup.find('h2', string='Anglais')
    if english_section:
        next_element = english_section.find_next_sibling()
        while next_element and next_element.name != 'h2':
            english_translations.append(next_element.text.strip())
            next_element = next_element.find_next_sibling()

    return french_translations, english_translations
    

# Main function to crawl French terms and their corresponding English translations, with pagination
def crawl_french_ai_terms(out_f):
    current_url = start_url
    
    while current_url:
        soup = get_soup(current_url)
        if not soup:
            break

        # Find all the terms on the current page
        for item in tqdm(soup.select('.mw-category-group a')):
            french_term = item.text.strip()
            term_url = base_url + item['href']
            
            
            french_translations, english_translations = get_translations(term_url)
            
            # Store the French term and its corresponding English translations
            terms = {
                'term_french': french_term,
                "terms_french": french_translations,
                'terms_english': english_translations,
            }
            json.dump(terms, out_f, ensure_ascii=False, indent=2)
            out_f.write("\n")
            out_f.flush()
        
        # Find the "page suivante" link to navigate to the next page
        next_page_link = soup.find('a', string='page suivante')
        if next_page_link:
            current_url = base_url + next_page_link['href']
        else:
            current_url = None  # No more pages, exit the loop
    

# Example usage
out_f = open("french.jsonl", 'a')
crawl_french_ai_terms(out_f)


100%|██████████| 36/36 [00:13<00:00,  2.59it/s]
